In [2]:
!pip install yt-dlp
!pip install ffmpeg-python==0.2.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.2/172.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.6 MB/s eta 0:00:00


In [3]:
import os
import yt_dlp
import shutil
import numpy as np
np.complex = complex
import librosa
import soundfile as sf
import random
from tqdm import tqdm

1. Download the songs

In [9]:
# Create output folder
output_dir = "MC_songs"
os.makedirs(output_dir, exist_ok=True)

# Access the links
song_links= []

with open("song_links.txt", "r", encoding="utf-8-sig") as f:
  for line in f:
        line = line.strip()
        parts = line.split()
        filename, url = parts[0].strip(), parts[1].strip()
        song_links.append((filename, url))

print(f"✅ Loaded {len(song_links)} Mandarin and Cantonese song links.")


# Set up yt-dlp
ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': os.path.join(output_dir, '%(title)s.%(ext)s'),
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
    'quiet': True
}


# Rename
for filename, url in song_links:
    print(f"⬇️ Downloading {filename} from {url}")
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        downloaded_name = ydl.prepare_filename(info).replace(".webm", ".mp3").replace(".m4a", ".mp3")
        target_name = os.path.join(output_dir, f"{filename}.mp3")
        os.rename(downloaded_name, target_name)
        print(f"✅ Saved as {target_name}")

✅ Loaded 160 Mandarin and Cantonese song links.
⬇️ Downloading Cantonese_001 from https://youtu.be/7qeShSmmsNg
✅ Saved as MC_songs/Cantonese_001.mp3
⬇️ Downloading Mandarin_001 from https://youtu.be/RIq7Tz9nYD8
✅ Saved as MC_songs/Mandarin_001.mp3
⬇️ Downloading Cantonese_002 from https://youtu.be/YX64gETkKrU
✅ Saved as MC_songs/Cantonese_002.mp3
⬇️ Downloading Mandarin_002 from https://youtu.be/7OdargV9K88
✅ Saved as MC_songs/Mandarin_002.mp3
⬇️ Downloading Cantonese_003 from https://youtu.be/ZFkAhKL0J2Q
✅ Saved as MC_songs/Cantonese_003.mp3
⬇️ Downloading Mandarin_003 from https://youtu.be/wTIcC-OF3mk
✅ Saved as MC_songs/Mandarin_003.mp3
⬇️ Downloading Cantonese_004 from https://youtu.be/IoxknYNHANU
✅ Saved as MC_songs/Cantonese_004.mp3
⬇️ Downloading Mandarin_004 from https://youtu.be/TEVbbBa_NHg
✅ Saved as MC_songs/Mandarin_004.mp3
⬇️ Downloading Cantonese_005 from https://youtu.be/7MEuDXdjbIE
✅ Saved as MC_songs/Cantonese_005.mp3
⬇️ Downloading Mandarin_005 from https://youtu.be/F

In [10]:
# Save as .zip
shutil.make_archive("MC_songs", 'zip', "MC_songs")

'/content/MC_songs.zip'

2. Clip

In [11]:
!unzip -q MC_songs.zip -d MC_songs

In [13]:
# Parameters
input_dir = "MC_songs"
output_dir = "MC_clips"
clip_duration = 30
clips_per_song = 5

os.makedirs(output_dir, exist_ok=True)

audio_files = [f for f in os.listdir(input_dir) if f.endswith((".wav", ".mp3"))]

for audio_file in audio_files:
    filepath = os.path.join(input_dir, audio_file)

    lower_name = audio_file.lower()
    if "mandarin" in lower_name:
        lang = "Mandarin"
    elif "cantonese" in lower_name:
        lang = "Cantonese"
    else:
        print(f"unidentified: {audio_file}")
        continue

    output_lang_dir = os.path.join(output_dir, lang)
    os.makedirs(output_lang_dir, exist_ok=True)

    try:
        y, sr = librosa.load(filepath, sr=None)
        total_duration = librosa.get_duration(y=y, sr=sr)
        max_start = total_duration - clip_duration

        if max_start <= 0:
            print(f"{audio_file}：inadequate")
            continue

        for i in range(clips_per_song):
            start_time = random.uniform(0, max_start)
            start_sample = int(start_time * sr)
            end_sample = start_sample + int(clip_duration * sr)
            clip_audio = y[start_sample:end_sample]

            clip_filename = f"{os.path.splitext(audio_file)[0]}_clip{i+1}.wav"
            clip_path = os.path.join(output_lang_dir, clip_filename)
            sf.write(clip_path, clip_audio, sr)

        print(f"completed:{audio_file} → {clips_per_song} clips in {lang}/")

    except Exception as e:
        print(f"error {audio_file}:{e}")

completed:Cantonese_044.mp3 → 5 clips in Cantonese/
completed:Cantonese_014.mp3 → 5 clips in Cantonese/
completed:Cantonese_036.mp3 → 5 clips in Cantonese/
completed:Mandarin_047.mp3 → 5 clips in Mandarin/
completed:Cantonese_002.mp3 → 5 clips in Cantonese/
completed:Cantonese_070.mp3 → 5 clips in Cantonese/
completed:Mandarin_010.mp3 → 5 clips in Mandarin/
completed:Mandarin_031.mp3 → 5 clips in Mandarin/
completed:Mandarin_071.mp3 → 5 clips in Mandarin/
completed:Cantonese_005.mp3 → 5 clips in Cantonese/
completed:Cantonese_080.mp3 → 5 clips in Cantonese/
completed:Cantonese_047.mp3 → 5 clips in Cantonese/
completed:Mandarin_076.mp3 → 5 clips in Mandarin/
completed:Mandarin_023.mp3 → 5 clips in Mandarin/
completed:Cantonese_031.mp3 → 5 clips in Cantonese/
completed:Mandarin_008.mp3 → 5 clips in Mandarin/
completed:Mandarin_066.mp3 → 5 clips in Mandarin/
completed:Mandarin_012.mp3 → 5 clips in Mandarin/
completed:Cantonese_055.mp3 → 5 clips in Cantonese/
completed:Cantonese_020.mp3 → 

3. Split the vocals

In [14]:
!pip install spleeter==2.4.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 585.9/585.9 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.8/82.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 kB 6.6 MB/s eta 0:0

In [1]:
import os
from spleeter.separator import Separator
from tqdm import tqdm

separator = Separator('spleeter:2stems')

input_root = "MC_clips"
output_root = "MC_vocals"

for lang in ["Mandarin", "Cantonese"]:
    input_dir = os.path.join(input_root, lang)
    output_dir = os.path.join(output_root, lang)
    os.makedirs(output_dir, exist_ok=True)

    for fname in tqdm(os.listdir(input_dir), desc=f"{lang}"):
        if fname.endswith(".wav"):
            input_path = os.path.join(input_dir, fname)
            temp_out = "/content/temp_sep"
            separator.separate_to_file(input_path, temp_out)

            vocals_path = os.path.join(temp_out, os.path.splitext(fname)[0], "vocals.wav")
            final_output_path = os.path.join(output_dir, fname)
            os.rename(vocals_path, final_output_path)

            os.system(f"rm -r '{temp_out}'")

Mandarin:   0%|          | 0/400 [00:00<?, ?it/s]

INFO:spleeter:Downloading model archive https://github.com/deezer/spleeter/releases/download/v1.4.0/2stems.tar.gz


INFO:spleeter:Downloading model archive https://github.com/deezer/spleeter/releases/download/v1.4.0/2stems.tar.gz


INFO:spleeter:Validating archive checksum


INFO:spleeter:Validating archive checksum


...

INFO:spleeter:File /content/temp_sep/Cantonese_068_clip2/accompaniment.wav written succesfully
Cantonese: 100%|█████████▉| 399/400 [1:11:35<00:10, 10.12s/it]

INFO:spleeter:File /content/temp_sep/Cantonese_050_clip2/vocals.wav written succesfully


INFO:spleeter:File /content/temp_sep/Cantonese_050_clip2/vocals.wav written succesfully


INFO:spleeter:File /content/temp_sep/Cantonese_050_clip2/accompaniment.wav written succesfully


INFO:spleeter:File /content/temp_sep/Cantonese_050_clip2/accompaniment.wav written succesfully
Cantonese: 100%|██████████| 400/400 [1:11:45<00:00, 10.76s/it]


In [4]:
shutil.make_archive("MC_vocals", 'zip', "MC_vocals")

'/content/MC_vocals.zip'

4. ➡️ ".npy" (Though in actual training we didn't use the .npy files directly from here.)

In [6]:
input_root = "MC_vocals"
output_root = "MC_mel_npy"

# .wav ➡️ Mel-Spectrogram ➡️ .npy
for lang in ["Cantonese", "Mandarin"]:
    input_dir = os.path.join(input_root, lang)
    output_dir = os.path.join(output_root, lang)
    os.makedirs(output_dir, exist_ok=True)

    for fname in tqdm(os.listdir(input_dir), desc=f"Processing {lang}"):
        if fname.endswith(".wav"):
            input_path = os.path.join(input_dir, fname)
            output_path = os.path.join(output_dir, fname.replace(".wav", ".npy"))

            y, sr = librosa.load(input_path, sr=22050)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=1024, hop_length=512, n_mels=128)
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

            np.save(output_path, mel_spec_db)

# Intergrating
mel_c_list = []
mel_m_list = []

for lang, target_list in [("Cantonese", mel_c_list), ("Mandarin", mel_m_list)]:
    input_dir = os.path.join(output_root, lang)
    for fname in sorted(os.listdir(input_dir)):
        if fname.endswith(".npy"):
            data = np.load(os.path.join(input_dir, fname))
            target_list.append(data)

mel_C = np.stack(mel_c_list)
mel_M = np.stack(mel_m_list)

# X.npy & y.npy
X = np.concatenate([mel_C, mel_M], axis=0)
y = np.concatenate([
    np.zeros(len(mel_M), dtype=np.int64),
    np.ones(len(mel_C), dtype=np.int64)
], axis=0)

np.save("X.npy", X)
np.save("y.npy", y)

print("X shape:", X.shape)
print("y shape:", y.shape)

Processing Mandarin: 100%|██████████| 400/400 [00:52<00:00,  7.66it/s]


X shape: (800, 128, 1292)
y shape: (800,)


5. Divide the data by proportion

In [7]:
from sklearn.model_selection import train_test_split

os.makedirs('MC_data', exist_ok=True)

X = np.load('X.npy')
y = np.load('y.npy')

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp)

np.save('MC_data/X_train.npy', X_train)
np.save('MC_data/y_train.npy', y_train)

np.save('MC_data/X_val.npy', X_val)
np.save('MC_data/y_val.npy', y_val)

np.save('MC_data/X_test.npy', X_test)
np.save('MC_data/y_test.npy', y_test)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (560, 128, 1292), Val: (120, 128, 1292), Test: (120, 128, 1292)


In [8]:
shutil.make_archive("MC_data", 'zip', "MC_data")

'/content/MC_data.zip'